# Personalized Recommendation System

This notebook ranks tracks from the user's listening history using listening behavior and context.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

processed = BASE_DIR / 'data' / 'processed'


In [ ]:
df = pd.read_csv(processed / 'spotify_clean_history.csv')
track = pd.read_csv(processed / 'spotify_track_behavior_features.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
df['skipped'] = df['skipped'].fillna(False).astype(bool)

print('Listening rows:', len(df))
print('Track rows:', len(track))


## 1. Build track recommendation features

A good candidate should have enough plays, a reasonable skip rate, and recent listening activity.

In [ ]:
last_play = (
    df.groupby(['master_metadata_track_name', 'master_metadata_album_artist_name'])['timestamp']
      .max()
      .reset_index(name='last_played')
)

recommendations = track.merge(
    last_play,
    on=['master_metadata_track_name', 'master_metadata_album_artist_name'],
    how='left'
)

recommendations['days_since_play'] = (
    df['timestamp'].max() - recommendations['last_played']
).dt.days

recommendations['days_since_play'] = recommendations['days_since_play'].fillna(9999)

print(recommendations.head())


## 2. Create a recommendation score

The score combines repeat listening, low skip rate, and recency. The weights are project-defined.

In [ ]:
recommendations['play_score'] = np.log1p(recommendations['total_plays'])
recommendations['skip_score'] = 1 - recommendations['skip_rate'].clip(0, 1)
recommendations['recency_score'] = 1 / (1 + recommendations['days_since_play'] / 30)

recommendations['recommendation_score'] = (
    0.50 * recommendations['play_score'] +
    0.30 * recommendations['skip_score'] +
    0.20 * recommendations['recency_score']
)

print(recommendations[['master_metadata_track_name', 'master_metadata_album_artist_name', 'recommendation_score']]
      .sort_values('recommendation_score', ascending=False)
      .head(10)
      .to_string(index=False))


## 3. Add a simple listening context

The context is based on when the user usually listens. This is a behavioral context, not automatic emotion detection.

In [ ]:
context = input('Choose context (night / weekend / general): ').strip().lower()

if context == 'night':
    context_rows = df[(df['hour'] >= 22) | (df['hour'] <= 5)]
elif context == 'weekend':
    context_rows = df[df['is_weekend'] == True]
else:
    context_rows = df

context_counts = (
    context_rows.groupby(['master_metadata_track_name', 'master_metadata_album_artist_name'])
    .size()
    .reset_index(name='context_plays')
)

recommendations = recommendations.merge(
    context_counts,
    on=['master_metadata_track_name', 'master_metadata_album_artist_name'],
    how='left'
)

recommendations['context_plays'] = recommendations['context_plays'].fillna(0)
recommendations['context_score'] = recommendations['context_plays'] / recommendations['total_plays'].clip(lower=1)

recommendations['final_score'] = (
    0.75 * recommendations['recommendation_score'] +
    0.25 * recommendations['context_score']
)



## 4. Generate recommendations

In [ ]:
result = (
    recommendations[
        ['master_metadata_track_name', 'master_metadata_album_artist_name',
         'total_plays', 'skip_rate', 'context_plays', 'final_score']
    ]
    .sort_values('final_score', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

result.index = result.index + 1
print(result.to_string())


## 5. Save recommendations

The result can later be displayed by the Streamlit application.

In [ ]:
output = processed / 'spotify_recommendations.csv'
result.to_csv(output, index=False)
print('Saved:', output)


## Final observations

- Recommendation type: behavior-based ranking
- Candidate tracks: tracks already present in the user's history
- Main signals: play count, skip rate, recency, and selected context
- Contexts tested: night, weekend, or general
- Important limitation: this notebook does not claim to detect a user's real emotions automatically.